# Rater Quality Evaluation

Measures how well each AI provider (Anthropic, OpenAI, Gemini) performed on the
subset of papers that have a known ground-truth screening decision.

Metrics computed per provider:
- **Cohen's Kappa** — agreement adjusted for chance
- **Sensitivity (Recall)** — % of true includes correctly identified
- **MCC** — Matthews Correlation Coefficient, robust to class imbalance

In [28]:
import json
import re
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    cohen_kappa_score,
    classification_report,
    matthews_corrcoef,
    recall_score,
)

RESULTS_DIR = Path("results")
GROUND_TRUTH_PATH = Path("papers/screening.csv")
CACHE_PATH = RESULTS_DIR / "processed.csv"

In [29]:
def extract_json_from_text(text: str) -> dict:
    """Extract JSON from plain text or markdown code block.

    Falls back to a targeted regex for truncated responses that never closed
    their markdown block (e.g. model hit a length limit mid-generation).
    """
    # Happy path: complete markdown code block
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    # Plain JSON (no code fence)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Truncated response: extract decision field directly
    m = re.search(r'"decision"\s*:\s*"(include|exclude)"', text)
    if m:
        return {"decision": m.group(1)}
    raise ValueError(f"Could not extract decision from text: {text[:100]!r}")


def normalize_reason(reason) -> list:
    """Ensure reason is always a list of strings."""
    if isinstance(reason, list):
        return reason
    if isinstance(reason, str):
        return [reason]
    return []


def parse_anthropic_line(line: str) -> dict | None:
    obj = json.loads(line)
    if obj.get("result", {}).get("type") != "succeeded":
        return None
    paper_id = int(obj["custom_id"])
    content = obj["result"]["message"]["content"]
    if not content:
        return None
    parsed = extract_json_from_text(content[0]["text"])
    return {
        "paper_id": paper_id,
        "provider": "anthropic",
        "decision": parsed["decision"],
        "reason": "|".join(normalize_reason(parsed.get("reason", []))),
    }


def parse_openai_line(line: str) -> dict | None:
    obj = json.loads(line)
    if obj.get("error") or obj["response"]["status_code"] != 200:
        return None
    paper_id = int(obj["custom_id"])
    text = obj["response"]["body"]["output"][0]["content"][0]["text"]
    parsed = extract_json_from_text(text)
    return {
        "paper_id": paper_id,
        "provider": "openai",
        "decision": parsed["decision"],
        "reason": "|".join(normalize_reason(parsed.get("reason", []))),
    }


def parse_gemini_line(line: str) -> dict | None:
    obj = json.loads(line)
    paper_id = int(obj["key"])
    candidates = obj.get("response", {}).get("candidates", [])
    if not candidates:
        return None
    text = candidates[0]["content"]["parts"][0]["text"]
    parsed = extract_json_from_text(text)
    return {
        "paper_id": paper_id,
        "provider": "gemini",
        "decision": parsed["decision"],
        "reason": "|".join(normalize_reason(parsed.get("reason", []))),
    }


PARSERS = {
    "anthropic": parse_anthropic_line,
    "openai": parse_openai_line,
    "gemini": parse_gemini_line,
}


def load_all_results() -> pd.DataFrame:
    if CACHE_PATH.exists():
        print(f"Loading from cache: {CACHE_PATH}")
        return pd.read_csv(CACHE_PATH)

    records = []
    for provider, parser in PARSERS.items():
        files = sorted(RESULTS_DIR.glob(f"{provider}-batch-*.jsonl"))
        for path in files:
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        record = parser(line)
                        if record:
                            records.append(record)
                    except Exception as e:
                        print(f"[{path.name}] Parse error: {e}")

    df = pd.DataFrame(records)
    df.to_csv(CACHE_PATH, index=False)
    print(f"Processed {len(df)} records -> saved to {CACHE_PATH}")
    return df


results = load_all_results()
print(f"\nRecords per provider:")
print(results.groupby("provider").size().to_string())

Loading from cache: results/processed.csv

Records per provider:
provider
anthropic    3793
gemini       3795
openai       3795


In [30]:
stats = (
    results.assign(doubt=results["reason"].str.contains("doubt", na=False))
    .groupby("provider")
    .agg(
        total=("paper_id", "count"),
        include=("decision", lambda x: (x == "include").sum()),
        exclude=("decision", lambda x: (x == "exclude").sum()),
        doubt=("doubt", "sum"),
    )
    .loc[["anthropic", "gemini", "openai"]]
)
stats["include_%"] = (stats["include"] / stats["total"] * 100).round(1)
stats["doubt_%"]   = (stats["doubt"]   / stats["total"] * 100).round(1)

print("Screening statistics — full dataset\n")
print(stats.to_string())

Screening statistics — full dataset

           total  include  exclude  doubt  include_%  doubt_%
provider                                                     
anthropic   3793      601     3192    139       15.8      3.7
gemini      3795      606     3187     43       16.0      1.1
openai      3795      458     3337    209       12.1      5.5


In [31]:
truth = pd.read_csv(GROUND_TRUTH_PATH)
truth = truth.rename(columns={"id": "paper_id", "decision": "truth"})
truth["truth_bin"] = (truth["truth"] == "include").astype(int)

print(f"Ground truth: {len(truth)} papers")
print(truth["truth"].value_counts().to_string())

Ground truth: 365 papers
truth
exclude    335
include     30


In [32]:
merged = results.merge(truth[["paper_id", "truth", "truth_bin"]], on="paper_id", how="inner")
merged["decision_bin"] = (merged["decision"] == "include").astype(int)

print("Coverage (papers with ground truth matched per provider):")
print(merged.groupby("provider").size().to_string())

Coverage (papers with ground truth matched per provider):
provider
anthropic    365
gemini       365
openai       365


In [33]:
providers = ["anthropic", "openai", "gemini"]
summary_rows = []

for provider in providers:
    subset = merged[merged["provider"] == provider]
    y_true = subset["truth_bin"]
    y_pred = subset["decision_bin"]

    kappa = cohen_kappa_score(y_true, y_pred)
    sensitivity = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    print(f"\n{'='*50}")
    print(f"Provider: {provider.upper()}  (n={len(subset)})")
    print(f"{'='*50}")
    print(f"  Kappa:       {kappa:.4f}")
    print(f"  Sensitivity: {sensitivity:.4f}")
    print(f"  MCC:         {mcc:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=["exclude", "include"]))

    summary_rows.append({
        "Provider": provider,
        "N": len(subset),
        "Kappa": round(kappa, 4),
        "Sensitivity": round(sensitivity, 4),
        "MCC": round(mcc, 4),
    })


Provider: ANTHROPIC  (n=365)
  Kappa:       0.3294
  Sensitivity: 1.0000
  MCC:         0.4440

              precision    recall  f1-score   support

     exclude       1.00      0.75      0.86       335
     include       0.26      1.00      0.42        30

    accuracy                           0.77       365
   macro avg       0.63      0.87      0.64       365
weighted avg       0.94      0.77      0.82       365


Provider: OPENAI  (n=365)
  Kappa:       0.4556
  Sensitivity: 1.0000
  MCC:         0.5431

              precision    recall  f1-score   support

     exclude       1.00      0.84      0.91       335
     include       0.35      1.00      0.52        30

    accuracy                           0.85       365
   macro avg       0.68      0.92      0.72       365
weighted avg       0.95      0.85      0.88       365


Provider: GEMINI  (n=365)
  Kappa:       0.3923
  Sensitivity: 1.0000
  MCC:         0.4939

              precision    recall  f1-score   support

     e

In [34]:
summary = pd.DataFrame(summary_rows).set_index("Provider")
print("\nSummary comparison:")
print(summary.to_string())


Summary comparison:
             N   Kappa  Sensitivity     MCC
Provider                                   
anthropic  365  0.3294          1.0  0.4440
openai     365  0.4556          1.0  0.5431
gemini     365  0.3923          1.0  0.4939


## Provider agreement analysis

Papers on which all three providers agree to exclude are strong candidates for automatic exclusion,
reducing the manual review workload. The cells below quantify the agreement structure across the
full dataset and validate each tier against the ground-truth subset.

In [35]:
# Pivot to one row per paper
pivot = results.pivot(index="paper_id", columns="provider", values="decision")
pivot["n_include"] = (pivot[["anthropic", "gemini", "openai"]] == "include").sum(axis=1)

tier_labels = {0: "All 3 exclude", 1: "1/3 include", 2: "2/3 include", 3: "All 3 include"}
pivot["tier"] = pivot["n_include"].map(tier_labels)

counts = pivot["tier"].value_counts().reindex(tier_labels.values())
counts_pct = (counts / len(pivot) * 100).round(1)

agreement = pd.DataFrame({"papers": counts, "%": counts_pct})
print("Agreement tiers — full dataset\n")
print(agreement.to_string())

Agreement tiers — full dataset

               papers     %
tier                       
All 3 exclude    2974  78.4
1/3 include       284   7.5
2/3 include       230   6.1
All 3 include     307   8.1


In [36]:
# Validate each tier against the ground truth
pivot_gt = pivot.merge(truth[["paper_id", "truth"]], on="paper_id", how="inner")

rows = []
for n, label in tier_labels.items():
    sub = pivot_gt[pivot_gt["n_include"] == n]
    true_inc = (sub["truth"] == "include").sum()
    rows.append({"tier": label, "gt_papers": len(sub), "true_includes": true_inc})

validation = pd.DataFrame(rows).set_index("tier")
print("Ground truth validation\n")
print(validation.to_string())

# Summary: how many papers need review under each strategy
print("\nManual review workload by strategy\n")
for threshold, label in [(3, "All 3 include only"), (2, ">=2/3 include"), (1, "Any include (>=1/3)")]:
    n_review = (pivot["n_include"] >= threshold).sum()
    gt_sub = pivot_gt[pivot_gt["n_include"] >= threshold]
    true_inc_covered = (gt_sub["truth"] == "include").sum()
    total_true_inc = (pivot_gt["truth"] == "include").sum()
    print(f"  {label:25s}: {n_review:4d} papers to review  "
          f"({n_review/len(pivot)*100:.1f}% of total)  "
          f"| true includes covered: {true_inc_covered}/{total_true_inc}")

Ground truth validation

               gt_papers  true_includes
tier                                   
All 3 exclude        226              0
1/3 include           44              0
2/3 include           32              0
All 3 include         63             30

Manual review workload by strategy

  All 3 include only       :  307 papers to review  (8.1% of total)  | true includes covered: 30/30
  >=2/3 include            :  537 papers to review  (14.2% of total)  | true includes covered: 30/30
  Any include (>=1/3)      :  821 papers to review  (21.6% of total)  | true includes covered: 30/30


## Discussion

The ground-truth subset contains 365 papers with a notable class imbalance: 30 includes (8.2%) and 335 excludes (91.8%). Under such imbalance, accuracy is an unreliable indicator of quality — a rater that excludes every paper would reach 91.8% accuracy. Cohen's Kappa and MCC are better suited here as they account for chance agreement and class distribution respectively.

### Sensitivity

All three providers achieved perfect sensitivity (1.0) on the ground-truth subset, meaning no true include was missed. In systematic review screening this property is generally prioritised over precision, as missing a relevant paper is considered a more serious error than retaining a false positive. The trade-off is over-inclusion: each provider flagged a number of papers that a human reviewer would exclude, requiring manual resolution.

### Agreement with ground truth

| Provider  | Kappa | Sensitivity | MCC   | Specificity | Include precision |
|-----------|-------|-------------|-------|-------------|-------------------|
| OpenAI    | 0.456 | 1.000       | 0.543 | 0.84        | 0.35              |
| Gemini    | 0.392 | 1.000       | 0.494 | 0.80        | 0.31              |
| Anthropic | 0.329 | 1.000       | 0.444 | 0.75        | 0.26              |

OpenAI showed the highest agreement with the ground truth across all metrics. Its specificity of 0.84 indicates that 84% of true excludes were correctly rejected, compared to 80% for Gemini and 75% for Anthropic. Kappa values range from 0.33 (Anthropic) to 0.46 (OpenAI), corresponding to *fair* to *moderate* agreement — a range that is not unusual for automated title/abstract screening on ambiguous corpora.

MCC ranges from −1 (perfect inverse) through 0 (no better than chance) to +1 (perfect agreement). All three providers score above zero (0.44–0.54), suggesting some predictive alignment with the ground truth beyond chance. Since sensitivity is identical across all three, the differences in MCC and Kappa are driven entirely by specificity — that is, by how conservatively each provider handles exclusions. Gemini (0.494) falls between OpenAI and Anthropic on this dimension.

### Provider agreement and review workload

Cross-provider agreement provides an additional signal for prioritising manual review. In the ground-truth subset, all 30 true includes fall in the tier where all three providers agreed to include — none appear in the 1/3 or 2/3 tiers. Under this pattern, restricting manual review to papers where all three providers agree to include (307 papers, 8.1% of the full set) would recover all known true includes while avoiding review of the remaining ~91.9%. Extending the threshold to ≥2/3 agreement (537 papers, 14.2%) adds a conservative buffer at the cost of additional review effort. These estimates carry uncertainty given the limited size of the ground-truth subset.

### Limitations

These results are based on a relatively small ground-truth subset (n=365), with only 30 positive examples, which limits the precision of sensitivity and agreement estimates. The metrics should be interpreted as indicative rather than definitive.

## Export for manual review

The agreement analysis shows that, on the ground-truth subset, all 30 true includes fall in the tier where all three providers agreed to include. No true include was found among papers where only one or two providers voted to include. Based on this, the 307 papers with unanimous inclusion form the candidate set for manual full-text review.

Papers already present in the ground-truth subset have a known decision and are excluded from the export. The remaining 244 papers are exported for manual adjudication.

In [37]:
unanimous_ids = pivot[pivot["n_include"] == 3].index

papers = pd.read_csv("papers/papers.csv", usecols=["id", "title", "abstract"])
export = (
    papers[papers["id"].isin(unanimous_ids)]
    .sort_values("id")
    [["id", "title", "abstract"]]
)

# Remove papers already decided in the ground truth
gt_ids = set(truth["paper_id"])
export = export[~export["id"].isin(gt_ids)].assign(final_decision="")

export_path = Path("papers/unanimous_include.csv")
export.to_csv(export_path, index=False)
print(f"Exported {len(export)} papers -> {export_path}")

Exported 244 papers -> papers/unanimous_include.csv
